# 07 — Survival Analysis: Reliability Engineering Framing

The most important notebook for reliability-engineering and warranty audiences.

Goal: Estimate the causal effect of the design variant on the TIME-TO-FAILURE distribution,
not just the average failure rate.

Stakeholder translation:
> 'What's the B10 life — the time at which 10% of units have failed — under old vs. new design?
> That's the warranty-exposure number a finance team can act on.'

Content:
1. Kaplan-Meier curves with log-rank test
2. Weibull AFT — shape and scale parameters (rho > 1 = wear-out pattern)
3. Cox PH with confounder adjustment — hazard ratio + CI
4. Schoenfeld residuals — PH assumption check
5. B10-life comparison table
6. Hazard-ratio forest plot (integrates with method comparison in notebook 08)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullAFTFitter
from lifelines.statistics import logrank_test
import sys; sys.path.insert(0, '..')
from src.diagnostics import plot_kaplan_meier
from src.estimators import cox_ph, weibull_aft

df = pd.read_csv('../data/field_panel.csv', parse_dates=['install_date'])
df['install_month_num'] = df.install_date.dt.month + (df.install_date.dt.year - 2022) * 12

%matplotlib inline

In [ ]:
# Kaplan-Meier curves
fig = plot_kaplan_meier(df, group_col='design_variant',
                        save_path='../figures/kaplan_meier.png')
plt.show()

In [ ]:
# B10-life: time at which 10% of units have failed (survival = 0.90)
fig, ax = plt.subplots(figsize=(8, 5))
for variant, color in [('A', 'tomato'), ('B', 'steelblue')]:
    sub = df[df.design_variant == variant]
    kmf = KaplanMeierFitter()
    kmf.fit(sub.time_to_event, sub.failure_event, label=f'Variant {variant}')
    kmf.plot_survival_function(ax=ax, color=color, ci_show=True)
    sf = kmf.survival_function_
    b10_month = (sf['KM_estimate'] <= 0.90).idxmax()
    ax.axvline(b10_month, color=color, linestyle=':', alpha=0.6)
    print(f'Variant {variant} B10 life: {b10_month} months')

ax.axhline(0.90, color='gray', linestyle='--', alpha=0.5, label='B10 threshold')
ax.set_title('B10-Life Comparison: Months to 10% Cumulative Failure')
ax.legend()
plt.savefig('../figures/b10_life.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cox Proportional Hazards — confounder-adjusted hazard ratio
CONFOUNDERS = ['region', 'install_crew', 'install_month_num']
result = cox_ph(df, CONFOUNDERS)
print('=== Cox PH Results ===')
print(f'Hazard Ratio (Variant B vs A): {result["hazard_ratio"]}')
print(f'95% CI: [{result["ci_lower"]}, {result["ci_upper"]}]')
print(f'p-value: {result["p_value"]}')
print(f'B10-life control: {result["b10_life_control_months"]} months')
print(f'B10-life treated: {result["b10_life_treated_months"]} months')
print(f'\nTrue HR to recover: 0.85')

In [ ]:
# Schoenfeld residuals — test proportional hazards assumption
df_enc = pd.get_dummies(df[['time_to_event', 'failure_event', 'install_month_num', 'operating_hours_month',
                              'region', 'install_crew', 'design_variant']], drop_first=True)
df_enc['treatment'] = (df['design_variant'] == 'B').astype(int)
cph = CoxPHFitter(penalizer=0.01)
cph.fit(df_enc, duration_col='time_to_event', event_col='failure_event')
cph.check_assumptions(df_enc, p_value_threshold=0.05)
print('\nIf p < 0.05 for any covariate: proportional hazards assumption may be violated')
print('Remedies: stratified Cox PH, or switch to accelerated failure time model')

In [ ]:
# Weibull AFT
print('=== Weibull AFT ===')
wf_result = weibull_aft(df)
wf_result['model'].print_summary()
print('\nInterpretation: rho > 1 → wear-out failure (increasing hazard with age = expected for HVAC)')
print('lambda_B vs. lambda_A → scale shift from design change')